# 01 · Criando o banco no SQL Server 2022

**Objetivo:** montar um banco que guarda embeddings **sem** o tipo `VECTOR` (que só chega no SQL Server 2025).

**A ideia em uma frase:** um embedding é uma lista de números decimais (float32). Cada float32 ocupa 4 bytes. Então um vetor de 384 dimensões vira **exatamente 1.536 bytes**, e isso cabe perfeitamente num `VARBINARY`.

Do SQL Server 2022 vamos aproveitar uma novidade do T-SQL: o `GENERATE_SERIES`, que permite ler esses bytes de volta **dentro do banco** (seção 9).

> Pré requisito: o arquivo `.env` da raiz do repositório existe e aponta para a sua instância de SQL Server 2022 (ou mais novo).

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=UserWarning)   # aviso do pandas sobre conexão pyodbc
sys.path.append(str(pathlib.Path.cwd().parent))   # permite "from src import ..."
from src import config, banco

## 1. Conferindo a versão: estamos mesmo no 2022?

In [2]:
conn_master = banco.conectar(autocommit=True)   # CREATE DATABASE exige autocommit
cur = conn_master.cursor()
print(cur.execute("SELECT @@VERSION").fetchval())

versao = int(cur.execute("SELECT SERVERPROPERTY('ProductMajorVersion')").fetchval())
print("Versão principal:", versao, "(16 = SQL Server 2022)")
assert versao >= 16, "Esta aula usa recursos do SQL Server 2022 (versão 16) ou mais novo."

Microsoft SQL Server 2022 (RTM-GDR) (KB5122771) - 16.0.1200.5 (X64) 
	Aug 20 2026 01:16:38 
	Copyright (C) 2022 Microsoft Corporation
	Developer Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)

Versão principal: 16 (16 = SQL Server 2022)


## 2. Prova de que o tipo VECTOR não existe aqui
Este erro é esperado. O 2022 trouxe muitas novidades, mas `VECTOR` e `VECTOR_DISTANCE` só chegam no 2025. É exatamente o problema que a aula resolve.

In [3]:
try:
    cur.execute("DECLARE @v VECTOR(3) = '[0.1, 0.2, 0.3]'; SELECT @v;")
except Exception as e:
    print("Como esperado, o 2022 não conhece VECTOR:\n", str(e)[:200])

Como esperado, o 2022 não conhece VECTOR:
 ('42000', "[42000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Column, parameter, or variable #1: Cannot find data type VECTOR. (2715) (SQLExecDirectW); [42000] [Microsoft][ODBC Driver 18 fo


## 3. Criando o banco

Um banco novo dentro da instância que já existe. O **nível de compatibilidade 160** é o do
SQL Server 2022: sem ele, o `GENERATE_SERIES` da seção 9 não funciona (bancos restaurados
de versões antigas costumam vir com um nível menor).

Quer mostrar a criação **do zero**, inclusive do banco? Coloque `RECRIAR_BANCO = True`.
Isso apaga tudo, então depois é preciso rodar o notebook 02 de novo para recarregar as
frases e os embeddings.

In [4]:
RECRIAR_BANCO = False   # True = apaga o banco inteiro e cria de novo

if RECRIAR_BANCO:
    banco.executar_script(conn_master, f'''
    IF DB_ID('{config.SQL_DATABASE}') IS NOT NULL
    BEGIN
        -- Derruba as conexões abertas (o Streamlit e outros notebooks seguram o banco)
        ALTER DATABASE [{config.SQL_DATABASE}] SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
        DROP DATABASE [{config.SQL_DATABASE}];
    END
    ''')
    print('Banco', config.SQL_DATABASE, 'apagado')

banco.executar_script(conn_master, f'''
IF DB_ID('{config.SQL_DATABASE}') IS NULL
    CREATE DATABASE [{config.SQL_DATABASE}];
GO
ALTER DATABASE [{config.SQL_DATABASE}] SET COMPATIBILITY_LEVEL = 160;   -- SQL Server 2022
''')
conn_master.close()

conn = banco.conectar(config.SQL_DATABASE, autocommit=True)
print('Conectado em', config.SQL_DATABASE)

Conectado em RagPoliticos


## 4. Zerando a aula: `DROP ... IF EXISTS` na ordem certa

Este notebook pode ser rodado **quantas vezes você quiser**: tudo o que ele cria, ele
apaga antes. É o que permite refazer a demonstração ao vivo sem erro de "objeto já existe"
— e sem o `IF ... IS NULL`, que esconderia justamente a parte que queremos mostrar: a criação.

A **ordem importa**, e ela é o inverso da criação:

1. procedures, view e função — ninguém depende delas;
2. `FraseEmbedding` antes de `Frase`, e `Frase` antes de `Politico` (a chave estrangeira
   filha sai primeiro, senão o SQL Server recusa o `DROP`);
3. o tipo `rag.Embedding`, que só pode sair depois da tabela que o usa;
4. por último o schema `rag`, que precisa estar vazio.

`DROP ... IF EXISTS` existe desde o SQL Server 2016; para schema e tipo o padrão é o mesmo.

In [5]:
banco.executar_script(conn, '''
-- 1) objetos programáveis: nada depende deles
DROP PROCEDURE IF EXISTS rag.usp_BuscaSemanticaTSQL;
DROP PROCEDURE IF EXISTS rag.usp_BuscaPalavraChave;
DROP VIEW IF EXISTS rag.vw_Frases;
DROP FUNCTION IF EXISTS rag.fn_VetorParaLinhas;

-- 2) tabelas: da filha para a mãe (por causa das chaves estrangeiras)
DROP TABLE IF EXISTS rag.FraseEmbedding;
DROP TABLE IF EXISTS rag.Frase;
DROP TABLE IF EXISTS rag.Politico;

-- 3) da aula antiga (2017): a tabela "explodida" não é mais usada
DROP TABLE IF EXISTS rag.FraseEmbeddingItem;

-- 4) o tipo só sai depois da tabela que o usava
DROP TYPE IF EXISTS rag.Embedding;

-- 5) e o schema, já vazio
DROP SCHEMA IF EXISTS rag;
''')

# Confirmando que não sobrou nada
import pandas as pd
pd.read_sql("""
SELECT COUNT(*) AS objetos_restantes
FROM sys.objects o JOIN sys.schemas s ON s.schema_id = o.schema_id
WHERE s.name = 'rag'
""", conn)

,objetos_restantes
0,0


## 5. Schema `rag`
Separa os objetos da aula do resto do banco.

In [6]:
banco.executar_script(conn, '''
CREATE SCHEMA rag;
''')

## 6. O nosso "tipo vetor" caseiro: `rag.Embedding`

Um **alias type** não muda o armazenamento, mas deixa a intenção explícita no modelo de dados. Quem abrir a tabela vai ler `Vetor rag.Embedding` e entender na hora.

Por que `VARBINARY` e não `NVARCHAR` com JSON? Veremos a comparação de tamanho no notebook 02.

In [7]:
banco.executar_script(conn, '''
-- Tipo de dado "apelido" (alias type): documenta a intenção da coluna.
-- Por baixo é VARBINARY(MAX): bytes crus de floats de 4 bytes.
CREATE TYPE rag.Embedding FROM VARBINARY(MAX) NULL;
''')

## 7. Tabelas de negócio: políticos e frases

In [8]:
banco.executar_script(conn, '''
CREATE TABLE rag.Politico (
    PoliticoId INT IDENTITY(1,1) CONSTRAINT PK_Politico PRIMARY KEY,
    Nome       NVARCHAR(150) NOT NULL CONSTRAINT UQ_Politico_Nome UNIQUE,
    Pais       NVARCHAR(60)  NOT NULL,
    Cargo      NVARCHAR(150) NULL
);
''')

In [9]:
banco.executar_script(conn, '''
CREATE TABLE rag.Frase (
    FraseId        INT IDENTITY(1,1) CONSTRAINT PK_Frase PRIMARY KEY,
    PoliticoId     INT NOT NULL CONSTRAINT FK_Frase_Politico REFERENCES rag.Politico(PoliticoId),
    Texto          NVARCHAR(1000) NOT NULL,   -- texto em português (o que vamos vetorizar)
    TextoOriginal  NVARCHAR(1000) NULL,       -- idioma original da frase
    IdiomaOriginal CHAR(2)        NULL,
    Ano            SMALLINT       NULL,
    Contexto       NVARCHAR(300)  NULL,
    Fonte          NVARCHAR(400)  NULL
);
''')

## 8. Tabela de embeddings

Três decisões de DBA aqui:
1. **Tabela separada**: trocar de modelo não altera a tabela de negócio.
2. **Chave (FraseId, Modelo)**: vetores de modelos diferentes **não são comparáveis**; o modelo faz parte da identidade.
3. **CHECK de tamanho**: `DATALENGTH(Vetor) = Dimensoes * 4` impede gravar lixo. É a governança que o banco oferece e a aplicação sozinha não garante.

In [10]:
banco.executar_script(conn, '''
-- Tabela separada para embeddings: permite ter mais de um modelo por frase
-- e re-vetorizar sem mexer na tabela de negócio.
CREATE TABLE rag.FraseEmbedding (
    FraseId    INT           NOT NULL CONSTRAINT FK_FraseEmb_Frase REFERENCES rag.Frase(FraseId),
    Modelo     NVARCHAR(200) NOT NULL,
    Dimensoes  SMALLINT      NOT NULL,
    Vetor      rag.Embedding NOT NULL,
    CriadoEm   DATETIME2(0)  NOT NULL CONSTRAINT DF_FraseEmb_CriadoEm DEFAULT SYSUTCDATETIME(),
    CONSTRAINT PK_FraseEmbedding PRIMARY KEY (FraseId, Modelo),
    -- Governança: garante que o tamanho em bytes bate com as dimensões (float32 = 4 bytes)
    CONSTRAINT CK_FraseEmb_Tamanho CHECK (DATALENGTH(Vetor) = Dimensoes * 4)
);
''')

## 9. Novidade do 2022: lendo o `VARBINARY` dentro do T-SQL

Para calcular o cosseno no banco, o T-SQL precisa enxergar os 384 números que estão dentro dos bytes. No 2017 isso exigia uma tabela auxiliar de números ou gravar o vetor "explodido" (uma linha por dimensão). No 2022, `GENERATE_SERIES(0, 383)` gera as posições na hora, e a função abaixo decodifica cada float32:

| float32 (IEEE 754) | sinal | expoente | mantissa |
|---|---|---|---|
| bits | 1 | 8 | 23 |

`valor = (-1)^sinal × (1 + mantissa / 2^23) × 2^(expoente − 127)`

Os bytes estão em *little-endian* (o byte menos significativo primeiro), por isso a função lê os 4 bytes de trás para frente.

In [11]:
banco.executar_script(conn, '''
-- Transforma um vetor VARBINARY (float32 little-endian) em linhas (Dim, Valor).
-- GENERATE_SERIES é novidade do SQL Server 2022 (nível de compatibilidade 160).
CREATE OR ALTER FUNCTION rag.fn_VetorParaLinhas (@Vetor VARBINARY(MAX))
RETURNS TABLE
AS RETURN
SELECT CAST(s.value AS SMALLINT) AS Dim,
       CASE WHEN f.Expoente = 0 THEN 0.0E0      -- zero (e subnormais, desprezíveis aqui)
            ELSE (1 - 2 * f.Sinal) * (1 + f.Mantissa / 8388608.0E0) * POWER(2.0E0, f.Expoente - 127)
       END AS Valor
FROM GENERATE_SERIES(0, CAST(DATALENGTH(@Vetor) / 4 - 1 AS INT)) AS s
-- 4 bytes invertidos (little-endian) -> inteiro de 32 bits sem sinal
CROSS APPLY (SELECT CAST(SUBSTRING(@Vetor, s.value * 4 + 4, 1) + SUBSTRING(@Vetor, s.value * 4 + 3, 1)
                       + SUBSTRING(@Vetor, s.value * 4 + 2, 1) + SUBSTRING(@Vetor, s.value * 4 + 1, 1) AS BIGINT) AS Bits) AS b
-- Separa os campos com divisão inteira: 8388608 = 2^23 e 256 = 2^8
CROSS APPLY (SELECT b.Bits / 8388608 / 256    AS Sinal,
                    (b.Bits / 8388608) % 256 AS Expoente,
                    b.Bits % 8388608         AS Mantissa) AS f;
''')

# Teste rápido com números conhecidos
from src.vetores import vetor_para_bytes
[round(r.Valor, 6) for r in conn.execute(
    "SELECT Valor FROM rag.fn_VetorParaLinhas(?) ORDER BY Dim", vetor_para_bytes([1.0, -2.5, 0.1])).fetchall()]

[1.0, -2.5, 0.1]

## 10. View e procedures

Aqui o `CREATE OR ALTER` faz o papel do drop: ele cria se não existir e substitui se existir.
É o padrão para objetos programáveis (view, função, procedure), que não guardam dados.

In [12]:
banco.executar_script(conn, '''
CREATE OR ALTER VIEW rag.vw_Frases AS
SELECT f.FraseId, p.Nome AS Politico, p.Pais, p.Cargo,
       f.Texto, f.TextoOriginal, f.IdiomaOriginal, f.Ano, f.Contexto, f.Fonte
FROM rag.Frase f
JOIN rag.Politico p ON p.PoliticoId = f.PoliticoId;
''')

In [13]:
banco.executar_script(conn, '''
-- Busca por palavra-chave: o jeito "clássico". Ignora acentos e maiúsculas (CI_AI).
CREATE OR ALTER PROCEDURE rag.usp_BuscaPalavraChave
    @Termo NVARCHAR(200)
AS
BEGIN
    SET NOCOUNT ON;
    SELECT FraseId, Politico, Texto, Ano
    FROM rag.vw_Frases
    WHERE Texto COLLATE Latin1_General_CI_AI LIKE N'%' + @Termo + N'%'
       OR TextoOriginal COLLATE Latin1_General_CI_AI LIKE N'%' + @Termo + N'%';
END;
''')

In [14]:
banco.executar_script(conn, '''
-- BÔNUS: busca semântica em T-SQL puro no SQL Server 2022, sem tipo VECTOR.
-- O vetor da pergunta chega em VARBINARY, no mesmo formato dos vetores gravados.
-- Como os vetores são normalizados, cosseno = soma dos produtos (produto escalar).
CREATE OR ALTER PROCEDURE rag.usp_BuscaSemanticaTSQL
    @Vetor  VARBINARY(MAX),
    @Modelo NVARCHAR(200),
    @K      INT = 5
AS
BEGIN
    SET NOCOUNT ON;
    WITH consulta AS (
        SELECT Dim, Valor FROM rag.fn_VetorParaLinhas(@Vetor)
    )
    SELECT TOP (@K) v.FraseId, v.Politico, v.Texto,
           SUM(e.Valor * c.Valor) AS Similaridade
    FROM rag.FraseEmbedding fe
    CROSS APPLY rag.fn_VetorParaLinhas(fe.Vetor) AS e
    JOIN consulta c ON c.Dim = e.Dim
    JOIN rag.vw_Frases v ON v.FraseId = fe.FraseId
    WHERE fe.Modelo = @Modelo
    GROUP BY v.FraseId, v.Politico, v.Texto
    ORDER BY Similaridade DESC;
END;
''')

## 11. Conferindo o que criamos

In [15]:
import pandas as pd
pd.read_sql("""
SELECT s.name AS [schema], o.name AS objeto, o.type_desc AS tipo
FROM sys.objects o JOIN sys.schemas s ON s.schema_id = o.schema_id
WHERE s.name = 'rag' AND o.type IN ('U','V','P','IF')
ORDER BY o.type_desc, o.name
""", conn)

,schema,objeto,tipo
0,rag,fn_VetorParaLinhas,SQL_INLINE_TABLE_VALUED_FUNCTION
1,rag,usp_BuscaPalavraChave,SQL_STORED_PROCEDURE
2,rag,usp_BuscaSemanticaTSQL,SQL_STORED_PROCEDURE
3,rag,Frase,USER_TABLE
4,rag,FraseEmbedding,USER_TABLE
5,rag,Politico,USER_TABLE
6,rag,vw_Frases,VIEW


In [16]:
pd.read_sql("""
SELECT c.name AS coluna, t.name AS tipo, bt.name AS tipo_base, c.max_length
FROM sys.columns c
JOIN sys.types t  ON t.user_type_id = c.user_type_id
JOIN sys.types bt ON bt.user_type_id = t.system_type_id
WHERE c.object_id = OBJECT_ID('rag.FraseEmbedding')
""", conn)

,coluna,tipo,tipo_base,max_length
0,FraseId,int,int,4
1,Modelo,nvarchar,nvarchar,400
2,Dimensoes,smallint,smallint,2
3,Vetor,Embedding,varbinary,-1
4,CriadoEm,datetime2,datetime2,6


Repare: a coluna `Vetor` aparece como tipo `Embedding`, com tipo base `varbinary` e `max_length = -1` (MAX).